The **AI Fact-Checking and Hallucination Detection System** is an **NLP-based application** that verifies the accuracy of AI-generated text. It accepts user input, divides it into individual claims, retrieves relevant information from Wikipedia, and compares each claim with the retrieved evidence using Sentence-BERT embeddings and Cosine Similarity. Based on the similarity score, the system classifies each claim as Verified, Partially Verified, or Hallucinated, helping users identify incorrect or misleading AI-generated information and improving the reliability of AI responses.

In [1]:
!pip install gradio
!pip install wikipedia-api
!pip install sentence-transformers
!pip install scikit-learn
!pip install nltk
!pip install torch

In [8]:
import re
import wikipediaapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Loading AI Model...")

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model Loaded Successfully!")

wiki = wikipediaapi.Wikipedia(
    language="en",
    user_agent="AI Fact Checker/1.0"
)

Loading AI Model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model Loaded Successfully!


In [9]:
def split_sentences(text):

    return re.split(r'(?<=[.!?])\s+', text.strip())


def extract_query(sentence):

    stop_words = {
        "the","is","are","was","were","of","in","on","at",
        "a","an","to","and","or","has","have","had",
        "located","created","around","by","from"
    }

    words = sentence.replace(".", "").split()

    keywords = []

    for word in words:

        if word.lower() not in stop_words:

            keywords.append(word)

    if len(keywords) >= 2:

        return keywords[0] + " " + keywords[1]

    elif len(keywords) == 1:

        return keywords[0]

    return sentence

In [10]:
def retrieve_fact(sentence):

    query = extract_query(sentence)

    page = wiki.page(query)

    if page.exists():

        return page.summary[:1500]

    return "No evidence found."


def similarity(claim, evidence):

    claim_embedding = model.encode([claim])

    evidence_embedding = model.encode([evidence])

    score = cosine_similarity(
        claim_embedding,
        evidence_embedding
    )[0][0]

    return score

In [11]:
def detect(claim, evidence):

    score = similarity(claim, evidence)

    if score >= 0.80:

        label = "Verified"

    elif score >= 0.60:

        label = "Partially Verified"

    else:

        label = "Hallucinated"

    return label, score


def fact_check(text):

    claims = split_sentences(text)

    scores = []

    print("=" * 80)
    print("FACT CHECK REPORT")
    print("=" * 80)

    for i, claim in enumerate(claims):

        if claim == "":
            continue

        evidence = retrieve_fact(claim)

        label, score = detect(claim, evidence)

        scores.append(score)

        print("\nClaim", i + 1)
        print("Text :", claim)
        print("Search :", extract_query(claim))
        print("Result :", label)
        print("Score :", round(score,3))
        print("Evidence :")
        print(evidence)
        print("-" * 80)

    if len(scores) > 0:

        average = sum(scores) / len(scores)

        print("\nOverall Confidence :", round(average,3))

In [12]:
print("\nAI FACT-CHECKING & HALLUCINATION DETECTION SYSTEM")

print("-" * 60)

print("Enter AI-generated text.")

print("Press Enter twice when finished.\n")

lines = []

while True:

    line = input()

    if line == "":

        break

    lines.append(line)

text = "\n".join(lines)

fact_check(text)




AI FACT-CHECKING & HALLUCINATION DETECTION SYSTEM
------------------------------------------------------------
Enter AI-generated text.
Press Enter twice when finished.

The Eiffel Tower is located in Berlin. Python was created by Guido van Rossum. The Sun revolves around Earth. India has 29 states.

FACT CHECK REPORT

Claim 1
Text : The Eiffel Tower is located in Berlin.
Search : Eiffel Tower
Result : Partially Verified
Score : 0.621
Evidence :
The Eiffel Tower (  EYE-fəl; French: Tour Eiffel [tuʁ ɛfɛl] ) is a lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889.
Locally nicknamed "La dame de fer" (French for "Iron Lady") for its use of wrought iron, it was constructed as the centrepiece of the 1889 World's Fair, and to crown the centennial anniversary of the French Revolution. Although initially criticised by some of France's leading artists and intellectuals for its design, i

Conclusion

The AI Fact-Checking and Hallucination Detection System provides an efficient way to verify AI-generated content by comparing user claims with trusted information from Wikipedia. Using Natural Language Processing (NLP), Sentence-BERT, and Cosine Similarity, the system classifies each claim as Verified, Partially Verified, or Hallucinated. This project helps improve the reliability and trustworthiness of AI-generated responses and can be further enhanced by integrating multiple knowledge sources and advanced AI models for higher accuracy.